In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
def check_if_gaps_exist(df):
    row_count_with_days_diff = 0
    df['reading_date'] = pd.to_datetime(df['reading_date'])
    df.sort_values(by=['patient_uuid','reading_date','batch'],inplace=True)
    for patient_uuid in df['patient_uuid'].unique():
        df_patient = df[df['patient_uuid']==patient_uuid]
        for batch in df_patient['batch'].unique():
            df_patient_batch = df_patient[df_patient['batch']==batch]
            df_patient_batch['reading_date'] = pd.to_datetime(df_patient_batch['reading_date'])
            df_patient_batch['next_reading_date'] = df_patient_batch['reading_date'].shift(-1)
            #print(int((vitals_hr_interpolated_patient_batch['next_reading_date'] - vitals_hr_interpolated_patient_batch['reading_date']).to_string().split(' ')[0]))
            df_patient_batch['days_difference'] = (df_patient_batch['next_reading_date'] - df_patient_batch['reading_date'])
            #print(df_patient_batch[(df_patient_batch['days_difference']>timedelta(days=1))& (df_patient_batch['days_difference']>timedelta(days=1))].shape[0])

            row_count_with_days_diff = int(df_patient_batch[(df_patient_batch['days_difference']>timedelta(days=1))& (df_patient_batch['days_difference']<timedelta(days=16))].shape[0])

            if row_count_with_days_diff > 0:
                print(df_patient_batch[(df_patient_batch['days_difference']>timedelta(days=1))& (df_patient_batch['days_difference']<timedelta(days=16))])
                break
    return row_count_with_days_diff

In [4]:
def extrapolate_into_batches(df):
    if {'SBP', 'HR'}.issubset(df.columns):
        final_columns = ['patient_uuid','reading_date','batch','HR','SBP']
    elif {'SBP'}.issubset(df.columns):
        final_columns = ['patient_uuid','reading_date','batch','SBP']
    elif {'HR'}.issubset(df.columns):
        final_columns = ['patient_uuid','reading_date','batch','HR']
    else:
        raise('required columns do not exist')
        
    if {'patient_uuid', 'reading_date'}.issubset(df.columns):
        pass
    else: 
        raise('required columns do not exist')
        
    df['reading_date'] = pd.to_datetime(df['reading_date'])
    df_minmaxtimes = pd.DataFrame(df.groupby('patient_uuid').agg({'reading_date' : np.min}))
    df_minmaxtimes.rename(columns={'reading_date':'reading_start_date'},inplace=True)

    df = pd.merge(df, df_minmaxtimes, on='patient_uuid')

    df['reading_date'] = pd.to_datetime(df['reading_date'])
    df['reading_start_date'] = pd.to_datetime(df['reading_start_date'])

    df.sort_values(by=['patient_uuid','reading_date'],inplace=True)

    df['days_elapsed'] = (df['reading_date'] - df['reading_start_date']).dt.days

    df['days_between_readings'] = -999

    for patient_uuid in df['patient_uuid'].unique():
        df.loc[df.patient_uuid == patient_uuid,'days_between_readings'] = (df[df.patient_uuid == patient_uuid]['days_elapsed'].shift(-1) - df[df.patient_uuid == patient_uuid]['days_elapsed'])



    df['batch'] = 0

    df.loc[df['days_between_readings'] > 1,'batch'] = 1

    for patient_uuid in df['patient_uuid'].unique():
        df.loc[df.patient_uuid == patient_uuid,'batch'] = df[df.patient_uuid == patient_uuid]['batch'].cumsum()

    df = df[final_columns]

    return df

In [5]:
vitals_hr_interpolated = pd.read_csv('vitals_hr_interpolated.csv')

vitals_sbp_interpolated = pd.read_csv('vitals_sbp_interpolated.csv')

vitals_hr_interpolated['reading_date'] = pd.to_datetime(vitals_hr_interpolated['reading_date'])

vitals_sbp_interpolated['reading_date'] = pd.to_datetime(vitals_sbp_interpolated['reading_date'])

In [6]:
vitals_hr_interpolated_extended = extrapolate_into_batches(vitals_hr_interpolated)

vitals_hr_interpolated_extended

,patient_uuid,reading_date,batch,HR
0,1048697,2019-08-26,0,80.000000
1,1048697,2019-08-27,0,92.666667
2,1048697,2019-08-28,0,84.500000
3,1048697,2019-08-29,0,84.333333
4,1048697,2019-08-30,0,91.500000
...,...,...,...,...
153957,99814066,2020-01-08,0,97.283923
153958,99814066,2020-01-09,0,93.460395
153959,99814066,2020-01-10,0,89.636867
153960,99814066,2020-01-11,0,85.813338


In [7]:
vitals_sbp_interpolated_extended = extrapolate_into_batches(vitals_sbp_interpolated)

vitals_sbp_interpolated_extended

,patient_uuid,reading_date,batch,SBP
0,1048697,2019-08-26,0,126.000000
1,1048697,2019-08-27,0,123.000000
2,1048697,2019-08-28,0,113.000000
3,1048697,2019-08-29,0,125.333333
4,1048697,2019-08-30,0,122.000000
...,...,...,...,...
152503,99814066,2020-01-08,0,112.820825
152504,99814066,2020-01-09,0,110.853570
152505,99814066,2020-01-10,0,108.886316
152506,99814066,2020-01-11,0,106.919062


In [8]:
check_if_gaps_exist(vitals_hr_interpolated_extended)

0

In [9]:
check_if_gaps_exist(vitals_sbp_interpolated_extended)

0

In [10]:
vitals_sbp_interpolated_invalid_patientids = (vitals_sbp_interpolated[(vitals_sbp_interpolated['SBP'] < 40) | (vitals_sbp_interpolated['SBP']>220)]).patient_uuid.unique()

In [11]:
vitals_sbp_interpolated = vitals_sbp_interpolated[~vitals_sbp_interpolated['patient_uuid'].isin(vitals_sbp_interpolated_invalid_patientids)]

In [12]:
vitals_sbp_interpolated = vitals_sbp_interpolated[vitals_sbp_interpolated['patient_uuid'].isin(vitals_hr_interpolated.patient_uuid.unique())]

In [13]:
check_if_gaps_exist(vitals_sbp_interpolated)

0

In [14]:
vitals_hr_interpolated_invalid_patientids = (vitals_hr_interpolated[(vitals_hr_interpolated['HR'] < 40) | (vitals_hr_interpolated['HR']>220)]).patient_uuid.unique()

In [15]:
vitals_hr_interpolated = vitals_hr_interpolated[~vitals_hr_interpolated['patient_uuid'].isin(vitals_hr_interpolated_invalid_patientids)]

In [16]:
vitals_hr_interpolated = vitals_hr_interpolated[vitals_hr_interpolated['patient_uuid'].isin(vitals_sbp_interpolated.patient_uuid.unique())]

In [17]:
check_if_gaps_exist(vitals_hr_interpolated)

0

In [18]:
vitals_interpolated = pd.merge(vitals_sbp_interpolated,vitals_hr_interpolated,on=['patient_uuid','reading_date'],how='inner')

In [19]:
vitals_interpolated_extended = extrapolate_into_batches(vitals_interpolated)

vitals_interpolated_extended

,patient_uuid,reading_date,batch,HR,SBP
0,1127912,2019-11-19,0,87.261064,124.853940
1,1127912,2019-11-20,0,92.000000,120.500000
2,1127912,2019-11-21,0,87.000000,131.750000
3,1127912,2019-11-22,0,89.000000,127.000000
4,1127912,2019-11-23,0,93.000000,109.500000
...,...,...,...,...,...
127630,99814066,2020-01-08,0,97.283923,112.820825
127631,99814066,2020-01-09,0,93.460395,110.853570
127632,99814066,2020-01-10,0,89.636867,108.886316
127633,99814066,2020-01-11,0,85.813338,106.919062


In [20]:
check_if_gaps_exist(vitals_interpolated_extended)

0

In [21]:
vitals_sbp_interpolated[vitals_sbp_interpolated['patient_uuid']==1519674]

,patient_uuid,reading_date,batch,SBP
608,1519674,2019-09-29,0,113.333333
609,1519674,2019-09-30,0,126.000000
610,1519674,2019-10-01,0,149.960448
611,1519674,2019-10-02,0,120.000000
612,1519674,2019-10-03,0,132.500000
...,...,...,...,...
708,1519674,2020-01-07,0,149.405370
709,1519674,2020-01-08,0,149.399706
710,1519674,2020-01-09,0,149.394042
711,1519674,2020-01-10,0,149.388378


In [22]:
vitals_hr_interpolated[vitals_hr_interpolated['patient_uuid']==1519674]

,patient_uuid,reading_date,batch,HR
613,1519674,2019-09-29,0,87.000000
614,1519674,2019-09-30,0,84.000000
615,1519674,2019-10-01,0,86.181835
616,1519674,2019-10-02,0,83.500000
617,1519674,2019-10-03,0,81.000000
...,...,...,...,...
713,1519674,2020-01-07,0,92.138495
714,1519674,2020-01-08,0,92.199277
715,1519674,2020-01-09,0,92.260060
716,1519674,2020-01-10,0,92.320842


In [ ]:
# Drops batches containing less than 30 observations.
df_recordings_per_patient = vitals_interpolated_extended.groupby(['patient_uuid', 'batch']).count().reset_index()
low_batches = df_recordings_per_patient[(df_recordings_per_patient.HR <= 30)][['patient_uuid','batch']]

for i in range(len(low_batches)):
    low_patient = low_batches.patient_uuid.iloc[i]
    low_batch = low_batches.batch.iloc[i]
    vitals_interpolated_extended.drop(vitals_interpolated_extended[(vitals_interpolated_extended.patient_uuid == low_patient) & (vitals_interpolated_extended.batch == low_batch)].index, inplace=True)


print(vitals_interpolated_extended.shape)
idx = vitals_interpolated_extended.groupby(['patient_uuid'])['batch'].transform(max) == vitals_interpolated_extended['batch']

vitals_interpolated_extended = vitals_interpolated_extended[idx]

print(vitals_interpolated_extended.shape)

In [23]:
vitals_interpolated_extended.to_csv("vitals_combined_interpolated.csv", index = False)

The cells below are some tests to check the quality of the combined, interpolated data

In [24]:
idx = vitals_interpolated_extended.groupby(['patient_uuid'])['batch'].transform(max) == vitals_interpolated_extended['batch']
vitals_interpolated_extended_maxbatch = vitals_interpolated_extended[idx]

In [25]:
vitals_interpolated_extended_maxbatch.groupby(['patient_uuid']).count()

,reading_date,batch,HR,SBP
patient_uuid,,,,
1127912,55,55,55,55
1164638,132,132,132,132
1166047,61,61,61,61
1193821,83,83,83,83
1231804,142,142,142,142
...,...,...,...,...
99553551,2,2,2,2
99591873,21,21,21,21
99683668,60,60,60,60


In [26]:
vitals_sbp_interpolated[(vitals_sbp_interpolated['SBP']<40)]

,patient_uuid,reading_date,batch,SBP


In [27]:
vitals_hr_interpolated[(vitals_hr_interpolated['HR']<40)]

,patient_uuid,reading_date,batch,HR


In [28]:
vitals_interpolated_extended[(vitals_interpolated_extended['SBP']<40)|(vitals_interpolated_extended['HR']<40) ]

,patient_uuid,reading_date,batch,HR,SBP


In [37]:
vitals_interpolated_extended_max_batches = pd.DataFrame(vitals_interpolated_extended.groupby(['patient_uuid']).agg({'batch' : np.max}))
vitals_interpolated_extended_max_batches['patient_uuid'] = vitals_interpolated_extended_max_batches.index
#vitals_interpolated_extended_max_batches.reset_index(level=[0])

vitals_interpolated_extended_max_batches[vitals_interpolated_extended_max_batches['batch']>3]

,batch,patient_uuid
patient_uuid,,
3581867,4,3581867
71277542,4,71277542
